In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [10]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [5]:
trainset = CIFAR10(root ="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root ="./data",train=False,download=True,transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:33<00:00, 5.09MB/s]


In [9]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

## Build the CNN

In [26]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel size =2 , stride =2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flatting
        x = self.fc_layer(x)

        return x

In [27]:
model = CNN()

In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [33]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)  # better practice than model.forward(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 0.42096581001339667
epoch = 2/10 & loss = 0.33172418790705066
epoch = 3/10 & loss = 0.26501025762551883
epoch = 4/10 & loss = 0.20345057338438072
epoch = 5/10 & loss = 0.1578916917671747
epoch = 6/10 & loss = 0.13333432930652672
epoch = 7/10 & loss = 0.11230562414373736
epoch = 8/10 & loss = 0.10052862054313463
epoch = 9/10 & loss = 0.0945425592770781
epoch = 10/10 & loss = 0.08049966885751146


In [34]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.31
